In [10]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [11]:
df = pd.read_csv("merged_data.csv.gz")

In [ ]:
def feature_engineering(df):
    # AMOUNT
    df['TransactionAmt_Log'] = np.log1p(df['TransactionAmt'])
    df['Amt_decimal'] = df['TransactionAmt'] % 1
    
    card1_group = df.groupby('card1')['TransactionAmt']
    df['card1_Amt_mean'] = card1_group.transform('mean')
    df['card1_Amt_std'] = card1_group.transform('std')
    
    df['AmountDeviationUser'] = df['TransactionAmt'] / (df['card1_Amt_mean'] + 0.001)
    df['AmountStdScore'] = (df['TransactionAmt'] - df['card1_Amt_mean']) / (df['card1_Amt_std'] + 0.001)
    df['AmountToMedianRatio'] = df['TransactionAmt'] / (df['card1_Amt_median'] + 0.001)
    
    amt_95 = df['TransactionAmt'].quantile(0.95)
    df['IsLargeTransaction'] = (df['TransactionAmt'] > amt_95).astype(int)
    df['IsSmallTestTransaction'] = (df['TransactionAmt'] < 5).astype(int)

    # TIME
    df['TransactionHour'] = (df['TransactionDT'] / 3600) % 24
    df['TransactionDayOfWeek'] = (df['TransactionDT'] / 86400) % 7
    df['IsNightTransaction'] = df['TransactionHour'].apply(lambda x: 1 if 0 <= x <= 5 else 0)
    df['IsWeekendTransaction'] = df['TransactionDayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
    df['TimeSinceLastTransaction'] = df.groupby('card1')['TransactionDT'].diff()

    df['temp_ts'] = pd.to_datetime(df['TransactionDT'], unit='s')
    df = df.set_index('temp_ts')
    df['TransactionVelocity1h'] = df.groupby('card1')['TransactionDT'].rolling('3600s').count().reset_index(level=0, drop=True)
    df['TransactionVelocity24h'] = df.groupby('card1')['TransactionDT'].rolling('86400s').count().reset_index(level=0, drop=True)
    df = df.reset_index(drop=True)
    
    # CARD
    df['CardTransactionCount'] = df.groupby('card1')['TransactionAmt'].transform('count')
    df['CardissuerFrequency'] = df['card1'].map(df['card1'].value_counts())

    df['DaysSinceRegistration'] = df['D1']
    df['AccountAgeRisk'] = 1 / (df['D1'] + 1)
    df['TimeSinceLastPurchase'] = df['D2'] 
    df['RegistrationToTransactionGap'] = df['TransactionDT'] - df['D1']
    df['D15_to_Mean_card1'] = df['D15'] / (df.groupby('card1')['D15'].transform('mean') + 0.001)

    # LOCATION
    df['AddrMismatch'] = (df['addr1'] != df['addr2']).astype(int)
    df['AddressTransactionCount'] = df.groupby('addr1')['TransactionAmt'].transform('count')
    df['CardAddressCombination'] = df.groupby(['card1', 'addr1'])['TransactionAmt'].transform('count')
    
    df['DistanceDeviation'] = df['dist1'] - df.groupby('card1')['dist1'].transform('mean')
    df['IsLongDistance'] = (df['dist1'] > 100).astype(int)
    df['DistanceRiskScore'] = (df['dist1'] - df['dist1'].min()) / (df['dist1'].max() - df['dist1'].min())
    df['card1_addr1_unique'] = df.groupby('card1')['addr1'].transform('nunique')

    # EMAIL
    df['EmailDomainMatch'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
    df['IsAnonymousEmail'] = df['P_emaildomain'].isin(['protonmail.com', 'mail.com']).astype(int)
    df['EmailDomainFrequency'] = df['P_emaildomain'].map(df['P_emaildomain'].value_counts())

    # DEVICE
    df['CardIPCount'] = df['C5']
    df['AddressDeviceCount'] = df['C7']
    df['AssociationRatio'] = (df['C1'] + df['C2']) / (df['C3'] + 0.01)
    df['TotalAssociations'] = df['C1'] + df['C2'] + df['C3']
    
    df['IsMobileDevice'] = (df['DeviceType'] == 'mobile').astype(int)
    df['null_counts'] = df.isnull().sum(axis=1)
    
    if 'id_31' in df.columns:
        df['id_31_device'] = df['id_31'].str.split(' ', expand=True)[0]
    df['Device_Freq'] = df['DeviceInfo'].map(df['DeviceInfo'].value_counts())
    df['C1_count'] = df['C1'].map(df['C1'].value_counts())

    
    v_cols = [c for c in df.columns if c.startswith('V')]
    if len(v_cols) > 0:
        pca = PCA(n_components=2)
        v_pca = pca.fit_transform(df[v_cols].fillna(-1))
        df['V_PCA_1'] = v_pca[:, 0]
        df['V_PCA_2'] = v_pca[:, 1]

    df = df.replace([np.inf, -np.inf], np.nan)
    
    return df

In [ ]:
feature_engineering(df)